In [0]:
%sql

SELECT * FROM 5_projects.dar053.mrn_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.person_alias_100samples_20260327 AS
WITH alias AS (
    SELECT
        TRY_CAST(MAX(PERSON_ID) AS STRING) AS person_id,
        ALIAS AS MRN
    FROM 4_prod.raw.MILL_PERSON_ALIAS 
    WHERE PERSON_ALIAS_TYPE_CD = 10 -- MRN
    GROUP BY ALIAS
)
SELECT alias.*
FROM alias
INNER JOIN (SELECT TRY_CAST(MRN AS STRING) AS MRN FROM 5_projects.dar053.mrn_100samples_20260327) AS m
ON LOWER(RTRIM(LTRIM(alias.MRN))) = LOWER(RTRIM(LTRIM(m.MRN)))


In [0]:
%sql
SELECT * FROM 5_projects.dar053.person_alias_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.rde_patient_demographics_100samples_20260327 AS
SELECT pd.PERSON_ID, Year_of_Birth, Gender, Ethnicity
FROM 4_prod.rde.rde_patient_demographics AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID

In [0]:
%sql
SELECT * FROM 5_projects.dar053.rde_patient_demographics_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.rde_all_diagnosis_100samples_20260327 AS
SELECT pd.* EXCEPT(MRN, NHS_Number)
FROM 4_prod.rde.rde_all_diagnosis AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID

In [0]:
%sql

SELECT * FROM 5_projects.dar053.rde_all_diagnosis_100samples_20260327 

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.rde_pathology_100samples_20260327 AS
SELECT pd.* EXCEPT(MRN, NHS_Number, Report, AnonReport)
FROM 4_prod.rde.rde_pathology AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID

In [0]:
%sql
SELECT * FROM 5_projects.dar053.rde_pathology_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.imaging_report_100samples_20260327 AS
SELECT pd.* EXCEPT (Mrn, Nhs_number, BlobContents, AnonymizedText) FROM 4_prod.pacs.imaging_report AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSONID = m.PERSON_ID
WHERE ExamCode IN (
  'UTECG', 'UTOEG', 'UTESD','MCSPS', 'MCARD','CACRY', 'CTAVI'
)


In [0]:
%sql
SELECT * FROM 5_projects.dar053.imaging_report_100samples_20260327


In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.imaging_report_texts_100samples_20260327 AS
SELECT ReportEventId,  b.AnonymizedText FROM 4_prod.pacs.imaging_report AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSONID = m.PERSON_ID
INNER JOIN (
  SELECT EventId, Max(AnonymizedText) AS AnonymizedText
  FROM 4_prod.rde.rde_blobdataset
  GROUP BY EventId
) AS b
ON pd.ReportEventId = b.EventId
WHERE pd.ExamCode IN (
  'UTECG', 'UTOEG', 'UTESD','MCSPS', 'MCARD','CACRY', 'CTAVI'
)

In [0]:
import os

base_path = "/Volumes/5_projects/dar053/texts"  # Or abfss:// for ADLS

# Function to write each row
def write_row(row):
    file_path = os.path.join(base_path, f"reporteventid={row['reporteventid']}.txt")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(row["anonymizedtext"])

df = spark.table("5_projects.dar053.imaging_report_texts_100samples_20260327")

# Apply function to each row without collecting
df.select("reporteventid", "anonymizedtext").foreach(write_row)

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.rde_pharmacyorders_100samples_20260327 AS
SELECT pd.* EXCEPT(MRN, NHS_Number)
FROM 4_prod.rde.rde_pharmacyorders AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID

In [0]:
%sql
SELECT * FROM 5_projects.dar053.rde_pharmacyorders_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.rde_encounter_100samples_20260327 AS
SELECT pd.* EXCEPT(MRN, NHS_Number)
FROM 4_prod.rde.rde_encounter AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID

In [0]:
%sql

SELECT * FROM 5_projects.dar053.rde_encounter_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.rde_measurements_100samples_20260327 AS
SELECT pd.* EXCEPT(MRN, NHS_Number)
FROM 4_prod.rde.rde_measurements AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID
WHERE EventType ILIKE '%height%' OR EventType ILIKE '%weight%'

In [0]:
%sql

SELECT * FROM 5_projects.dar053.rde_measurements_100samples_20260327

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar053.imd_100samples_20260327 AS
SELECT pd.PERSON_ID, pd.imd_decile
FROM 4_prod.rde.imd AS pd
INNER JOIN (
  SELECT CAST(CAST(PERSON_ID AS DOUBLE) AS BIGINT) AS PERSON_ID
  FROM 5_projects.dar053.person_alias_100samples_20260327
) AS m
ON pd.PERSON_ID = m.PERSON_ID

In [0]:
%sql
SELECT * FROM 5_projects.dar053.imd_100samples_20260327